#### Application Codebase Knowledge Graph (KG) Tools

##### Environment setup

###### Package imports

In [119]:
import os
import sys
from dotenv import load_dotenv
import time
import requests
import pandas as pd
from pathlib import Path
import importlib
from neo4j import GraphDatabase
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget
import gradio as gr
import gradio.blocks

###### Test connections

In [120]:
def test_neo4j_connection(env_file=Path.cwd().parent / ".env"):
    """
    Test connectivity to a Neo4j database using settings
    stored in a .env file.

    Expected variables:

        NEO4J_URI
        NEO4J_USERNAME
        NEO4J_PASSWORD
        NEO4J_DATABASE
    """

    # Load environment variables
    load_dotenv(env_file, override=True)

    uri = os.getenv("NEO4J_URI")
    username = os.getenv("NEO4J_USERNAME")
    password = os.getenv("NEO4J_PASSWORD")
    database = os.getenv("NEO4J_DATABASE")

    try:
        with GraphDatabase.driver(uri, auth=(username, password), database=database) as driver:

            # Test the connection
            driver.verify_connectivity()

            # Open a session
            with driver.session(database=database) as session:
                result = session.run(
                    """
                    RETURN
                        1 AS connected,
                        datetime() AS server_time
                    """
                )

                record = result.single()

                print("✅ Successfully connected to Neo4j")
                print(f"Database    : {database}")
                print(f"URI         : {uri}")
                print(f"Server Time : {record['server_time']}")

                return True

    except Exception as ex:
        print("❌ Connection failed")
        print(type(ex).__name__)
        print(ex)

        return False

test_neo4j_connection()

✅ Successfully connected to Neo4j
Database    : appkg
URI         : neo4j://127.0.0.1:7687
Server Time : 2026-09-18T11:44:30.970000000+00:00


True

###### Create Neo4j driver object and python-cypher method

In [121]:
# Load .env variables
uri = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
database = os.getenv("NEO4J_DATABASE")

# Create driver object
# Keeping the Neo4j Driver open for the lifetime of your Jupyter session is the normal pattern and does not, by itself, constitute a memory leak.
driver = GraphDatabase.driver(uri, auth=(username, password), database=database)

driver.verify_connectivity()

# Define a generic function to run a Cypher query
def run_cypher_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return result.data()

###### Test Github API connection

In [122]:
def test_github_connection():
    """Test connectivity to GitHub and validate the GitHub PAT."""

    # Find .env in the directory above the notebook's working directory
    env_file = Path.cwd().parent / ".env"

    print(f"Loading environment from: {env_file}")

    if not env_file.exists():
        print("❌ .env file not found")
        return False

    load_dotenv(env_file, override=True)

    token = os.getenv("GITHUB_TOKEN")

    if not token:
        print("❌ GITHUB_TOKEN is missing from .env")
        return False

    # GitHub API headers
    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": "2026-03-10",
    }

    try:
        # Test GitHub API + PAT authentication
        response = requests.get(
            "https://api.github.com/user",
            headers=headers,
            timeout=10,
        )

        print(f"GitHub API status: {response.status_code}")

        if response.status_code == 200:
            user = response.json()

            print("✅ Successfully connected to GitHub")
            print("✅ PAT is valid")
            print(f"GitHub user : {user.get('login')}")
            print(f"User ID     : {user.get('id')}")
            print(f"Account type: {user.get('type')}")

            return True

        elif response.status_code == 401:
            print("❌ GitHub API connection succeeded")
            print("❌ PAT authentication failed")
            print("The token may be invalid, expired, or revoked.")

        elif response.status_code == 403:
            print("⚠️ GitHub API responded with 403 Forbidden")
            print("The PAT was received, but GitHub denied the request.")
            print(response.text)

        else:
            print("❌ GitHub API request failed")
            print(response.text)

        return False

    except requests.exceptions.Timeout:
        print("❌ Connection to GitHub timed out")
        return False

    except requests.exceptions.ConnectionError as ex:
        print("❌ Could not connect to GitHub")
        print(ex)
        return False

    except requests.exceptions.RequestException as ex:
        print("❌ GitHub request failed")
        print(type(ex).__name__)
        print(ex)
        return False

test_github_connection()

Loading environment from: E:\Projects\knowledge_graph_tools\.env
GitHub API status: 200
✅ Successfully connected to GitHub
✅ PAT is valid
GitHub user : georgejaymcmc
User ID     : 88515517
Account type: User


True

##### Repo details

In [146]:
# 1. Enable auto-reloading so changes to your .py file update automatically
%load_ext autoreload
%autoreload 2

# 2. Add your project 'src' directory to Python's path
# Adjust the number of parents depending on where your notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parents[0] # Assumes notebook is one folder deep (e.g., in a 'notebooks/' folder)
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 3. Import your service directly from your local module
from codebase_kg.services.github_repo_service import GithubRepoService

# 4. Initialize and use it
# service = GithubRepoService(token="YOUR_GITHUB_TOKEN")
print("Module successfully imported from:", sys.modules['codebase_kg.services.github_repo_service'].__file__)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Module successfully imported from: E:\Projects\knowledge_graph_tools\src\codebase_kg\services\github_repo_service.py


##### Extracting Zotero repo stats

###### Code: Choose repo to analyse
1. Fork repo to baseline codebase - 16/09/2026

In [147]:
# Load Github PAT
env_file = Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

github_token = os.getenv("GITHUB_TOKEN")

# Create the service
github = GithubRepoService(token=github_token)

# Repository name
owner = "georgejaymcmc"
repo = "zotero"

# Print result
result = github.stats(owner, repo)

# Pretty print result
for key, value in result.items():
    print(f"{key:20}: {value}")


name                : zotero
full_name           : georgejaymcmc/zotero
description         : Zotero is a free, easy-to-use tool to help you collect, organize, annotate, cite, and share your research sources.
language            : None
size_kb             : 242678
visibility          : public
default_branch      : main
stars               : 0
forks               : 0
watchers            : 0
directories         : 363
files               : 3489
url                 : https://github.com/georgejaymcmc/zotero


###### Code: Purge the cached module from the notebook

In [148]:
# 1. Collect all loaded submodules related to your project package
to_delete = [name for name in sys.modules if name.startswith("codebase_kg")]

# 2. Purge them from the sys.modules memory cache
for module_name in to_delete:
    del sys.modules[module_name]
    print(f"🔥 Purged cache for: {module_name}")

# 3. Re-import your updated class freshly from the disk
from codebase_kg.services.github_repo_service import GithubRepoService


🔥 Purged cache for: codebase_kg
🔥 Purged cache for: codebase_kg.services
🔥 Purged cache for: codebase_kg.services.github_repo_service


###### Code: Output repo file types and directory location to csv

In [149]:
# Load Github PAT
env_file = Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

github_token = os.getenv("GITHUB_TOKEN")

# Create the service
github = GithubRepoService(token=github_token)

# Repository name
owner = "georgejaymcmc"
repo = "zotero"


In [150]:
# 1. RUN THIS METHOD TO GENERATE THE CSV FILE
print("Generating CSV file...")
csv_results = github.file_types(owner, repo, output_csv="zotero_files.csv")

# 2. Pretty print the execution summary returned by file_types
print("\n--- Execution Summary ---")
for key, value in csv_results.items():
    print(f"{key:20}: {value}")


Generating CSV file...
📦 Dataframe successfully dumped to: E:\Projects\knowledge_graph_tools\src\codebase_kg\neo4j_imports\zotero_files.csv

--- Execution Summary ---
total_files         : 3489
directory_count     : 271
output_csv          : zotero_files.csv
file_type_counts    : {'SVG': 774, 'FreeMarker_Java_template': 673, 'Javascript': 578, 'DTD_XML_def': 482, 'CSS': 202, 'Java_prop': 146, 'Other': 108, 'Header': 107, 'C++': 77, 'JSON': 63, 'XHTML': 51, 'PNG': 41, 'HTML': 23, 'PDF': 18, 'Text': 16, 'Typescript': 14, 'Shell': 12, 'IDL': 10, 'NSH': 7, 'GIF': 7, 'RSS': 7, 'Epub': 7, 'Markdown': 6, 'RC': 6, 'XUL': 6, 'NSI': 5, 'XML': 4, 'Python Script': 4, 'ICO': 4, 'SQL': 4, '.ini': 3, '.manifest': 3, 'CSL': 3, 'VBScript': 2, 'WOFF': 2, 'SQLite': 2, 'RDF': 2, 'YAML': 1, 'C': 1, 'NLF': 1, 'XPI': 1, 'ATOM': 1, 'OPML': 1, 'LUA': 1, 'OPF': 1, 'JPG': 1, 'ZIP': 1}
unknown_extensions  : ['', '.0_release_build_and_deploy', '.car', '.ctv6', '.def', '.desktop', '.dsp', '.dsw', '.mts', '.patch', 

##### Import what we know into Neo4j: KGs are built progressively
- directories, file names and file types form the base of the App KG

###### Start from scratch: Delete all nodes, relationships, etc from Neo4j

In [142]:
# Delete all nodes and relationships

query_delete_all_nodes_relationships = """
MATCH (n)
DETACH DELETE n
RETURN count(n) AS deleted_nodes
"""

result_query_delete_all_nodes_relationships = run_cypher_query(query_delete_all_nodes_relationships)

result_query_delete_all_nodes_relationships

[{'deleted_nodes': 0}]

###### Upload repo_files.csv into Neo4j to create base graph elements

In [151]:
# Generic function to run Cypher queries
query_load_repo_files = """
// Load File Type, Directory, and File Name nodes
LOAD CSV WITH HEADERS FROM 'file:///zotero_files.csv' AS row
MERGE (f:FileName {name: row.FileName})
MERGE (d:Directory {name: row.Directory})
MERGE (t:FileType {type: row.FileType})
MERGE (d)-[:CONTAINS]->(f)
MERGE (f)-[:IS_TYPE_OF]->(t);
"""
result_query_load_repo_files = run_cypher_query(query_load_repo_files)

###### Check the upload
- Compare file count with above repo query result

In [152]:
# Generate graph from Cypher query
g = Neo4jGraphWidget(driver)
# View schema
g.show_cypher("CALL db.schema.visualization()")

query_node_frequency = """
MATCH (n)
RETURN labels(n) AS label, count(*) AS count
ORDER BY count DESC;
"""
result_query_node_frequency = run_cypher_query(query_node_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_nodes = pd.DataFrame(result_query_node_frequency)
# Flatten the 'label' column since it's a list
df_nodes['label'] = df_nodes['label'].apply(
    lambda x: x[0] if len(x) > 0 else None)
print("Nodes by Count")
print(df_nodes)  # Outputs the list of nodes (Persons)

query_relationship_frequency = """
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, COUNT(r) AS count
ORDER BY count DESC;
"""
result_query_relationship_frequency = run_cypher_query(
    query_relationship_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_rels = pd.DataFrame(result_query_relationship_frequency)
# Flatten the 'label' column since it's a list
# df_rels['label'] = df_rels['label'].apply(lambda x: x[0] if len(x) > 0 else None)
print("Relationships by Count")
print(df_rels)  # Outputs the list of nodes (Persons)


GraphWidget(layout=Layout(height='500px', width='100%'))

Nodes by Count
       label  count
0   FileName   1634
1  Directory    271
2   FileType     47
Relationships by Count
  relationshipType  count
0         CONTAINS   3489
1       IS_TYPE_OF   1634


##### Gradio chatbot file selector

###### Function to list labels and relationship types

In [132]:
def get_graph_schema():
    """Return node labels and relationship types with counts as a DataFrame."""

    try:

        # ---------------------------------
        # Node labels and properties
        # ---------------------------------

        labels = run_cypher_query("""
            MATCH (n)
            UNWIND labels(n) AS label
            UNWIND keys(n) AS property
            RETURN
                label,
                count(DISTINCT n) AS count,
                collect(DISTINCT property) AS properties
            ORDER BY label
        """)

        # ---------------------------------
        # Relationship types
        # ---------------------------------

        relationships = run_cypher_query("""
            MATCH ()-[r]->()
            RETURN
                type(r) AS relationship,
                count(*) AS count
            ORDER BY relationship
        """)

        # ---------------------------------
        # Build combined result
        # ---------------------------------

        rows = []

        # Nodes
        for row in labels:
            rows.append({
                "Type": "Node",
                "Name": row["label"],
                "Count": row["count"],
                "Properties": ", ".join(
                    sorted(row["properties"])
                )
            })

        # Relationships
        for row in relationships:
            rows.append({
                "Type": "Relationship",
                "Name": row["relationship"],
                "Count": row["count"],
                "Properties": ""
            })

        # ---------------------------------
        # Return DataFrame
        # ---------------------------------

        return pd.DataFrame(
            rows,
            columns=[
                "Type",
                "Name",
                "Count",
                "Properties"
            ]
        )

    except Exception as e:

        return pd.DataFrame({
            "Error": [
                f"{type(e).__name__}: {e}"
            ]
        })

###### Convert cypher query results to pandas df

In [133]:
def execute_cypher(query):
    """Execute Cypher and return the results as a DataFrame."""

    if not query or not query.strip():
        return pd.DataFrame({
            "Result": ["Please enter a Cypher query."]
        })

    try:
        records = run_cypher_query(query)

        if not records:
            return pd.DataFrame({
                "Result": ["Query returned no rows."]
            })

        return pd.DataFrame(records)

    except Exception as e:
        return pd.DataFrame({
            "Error": [
                f"{type(e).__name__}: {e}"
            ]
        })

###### Gradio interface

###### Test queries

In [134]:
query_test="""
MATCH (d:Directory)-[:CONTAINS]->(f:FileName)-[:IS_TYPE_OF]->(ft:FileType)
WHERE ft.type = "Javascript"

WITH f.name AS file_name, count(DISTINCT d) AS directory_count
WHERE directory_count > 1

MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
WHERE f.name = file_name

RETURN
    file_name,
    d.name AS directory
ORDER BY file_name, directory
"""

result_query_test = run_cypher_query(query_test)

result_query_test

[{'file_name': 'commandLineHandler.js', 'directory': 'app/assets'},
 {'file_name': 'commandLineHandler.js',
  'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'contextPane.js', 'directory': 'chrome/content/zotero'},
 {'file_name': 'contextPane.js',
  'directory': 'chrome/content/zotero/elements'},
 {'file_name': 'http.js',
  'directory': 'chrome/content/zotero/actors/translation'},
 {'file_name': 'http.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'id.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'id.js', 'directory': 'resource/tinymce/langs'},
 {'file_name': 'locale.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'locale.js', 'directory': 'resource/tinymce'},
 {'file_name': 'plugin.min.js',
  'directory': 'resource/tinymce/plugins/autolink'},
 {'file_name': 'plugin.min.js', 'directory': 'resource/tinymce/plugins/code'},
 {'file_name': 'plugin.min.js',
  'directory': 'resource/tinymce/plugins/contextmenu'},
 {'file_name'

In [135]:

with gr.Blocks(title="Neo4j Cypher Explorer") as demo:
    gr.Markdown("# Neo4j Cypher Explorer")
    gr.Markdown(
        "Explore the current graph model and execute Cypher queries."
    )

    # ---------------------------------------------------------
    # Graph schema
    # ---------------------------------------------------------

    gr.Markdown("## Graph Schema")

    schema_button = gr.Button(
        "Refresh Graph Schema",
        variant="secondary"
    )

    schema_output = gr.Dataframe(
        label="Graph Schema",
        interactive=False,
        wrap=True
    )

    # ---------------------------------------------------------
    # Cypher query
    # ---------------------------------------------------------

    gr.Markdown("## Cypher Query")

    cypher_input = gr.Textbox(
        label="Enter Cypher",
        lines=8,
        value="""
        MATCH (d:Directory)-[:CONTAINS]->(f:FileName)-[:IS_TYPE_OF]->(ft:FileType)
WHERE ft.type = "Javascript"

WITH f.name AS file_name, count(DISTINCT d) AS directory_count
WHERE directory_count > 1

MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
WHERE f.name = file_name

RETURN
    file_name,
    d.name AS directory
ORDER BY file_name, directory
    """
    )

    execute_button = gr.Button(
        "Run Query",
        variant="primary"
    )

    # ---------------------------------------------------------
    # Results
    # ---------------------------------------------------------

    gr.Markdown("## Results")

    result_output = gr.Dataframe(
        label="Query Result",
        interactive=False,
        wrap=True
    )

    # ---------------------------------------------------------
    # Events
    # ---------------------------------------------------------

    schema_button.click(
        fn=get_graph_schema,
        outputs=schema_output
    )

    execute_button.click(
        fn=execute_cypher,
        inputs=cypher_input,
        outputs=result_output
    )

    cypher_input.submit(
        fn=execute_cypher,
        inputs=cypher_input,
        outputs=result_output
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


##### Display repo file contents
```mermaid
sequenceDiagram
    participant A as Python CLI
    participant B as NEO4J
    participant C as GITHUB

    A->>B: 1. Query Neo4j for list of files
    B->>A: 2. List files in a Gradio SelectBox
    A->>C: 3. Select file to get syntax
    C->>A: 4. Return file contents for display in Gradio
```

###### 1. Query Neo4j for files

In [162]:
def get_files_from_neo4j(file_type=None):
    """Return files from Neo4j, optionally filtered by file type."""

    query = """
    MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
          -[:IS_TYPE_OF]->(ft:FileType)

    WHERE $file_type IS NULL
       OR ft.type = $file_type

    RETURN
        d.name AS directory,
        f.name AS file_name,
        ft.type AS file_type

    ORDER BY directory, file_name
    """

    records = run_cypher_query(
        query,
        {"file_type": file_type}
    )

    return pd.DataFrame(records)

###### 2. Create Gradio file choices

In [157]:
def build_github_path(directory, file_name):
    """Convert Neo4j directory/file information into a GitHub path."""

    directory = str(directory).strip()

    if directory.upper() == "ROOT":
        return file_name

    return f"{directory}/{file_name}"


def get_file_choices(file_type=None):
    """Build Gradio file choices from Neo4j."""

    df = get_files_from_neo4j(file_type)

    if df.empty:
        return []

    choices = []

    for _, row in df.iterrows():

        file_path = build_github_path(
            row["directory"],
            row["file_name"]
        )

        choices.append((file_path, file_path))

    return choices

In [158]:
def get_file_types():
    """Return file types available in the Neo4j codebase."""

    query = """
    MATCH (ft:FileType)
    RETURN DISTINCT ft.type AS file_type
    ORDER BY file_type
    """

    records = run_cypher_query(query)

    return [
        row["file_type"]
        for row in records
        if row["file_type"]
    ]

In [167]:
def update_file_dropdown(file_type):
    """Update the file dropdown based on the selected file type."""

    if not file_type:
        return gr.Dropdown(
            choices=[],
            value=None
        )

    choices = get_file_choices(file_type)

    return gr.Dropdown(
        choices=choices,
        value=None
    )

###### 3. Map Neo4j FileType to `gr.Code` languages

In [154]:
from pathlib import Path


def get_language(file_name):
    """Return the Gradio language for a filename."""

    extension = Path(file_name).suffix.lower()

    language_map = {
        ".js": "javascript",
        ".jsx": "javascript",
        ".ts": "typescript",
        ".tsx": "typescript",

        ".py": "python",

        ".json": "json",

        ".html": "html",
        ".htm": "html",

        ".css": "css",

        ".sql": "sql",

        ".java": "java",

        ".c": "c",
        ".h": "c",
        ".cpp": "cpp",
        ".hpp": "cpp",

        ".cs": "csharp",

        ".md": "markdown",

        ".xml": "xml",

        ".yaml": "yaml",
        ".yml": "yaml",

        ".sh": "shell",

        ".txt": "text",
    }

    return language_map.get(extension, "text")

###### 4. Retrieve code from Github

In [186]:
def get_selected_file(file_path):
    """Retrieve the selected file from GitHub."""

    if not file_path:
        return "", "text"

    df = get_files_from_neo4j()

    matches = df[
        df.apply(
            lambda row: build_github_path(
                row["directory"],
                row["file_name"]
            ) == file_path,
            axis=1
        )
    ]

    if matches.empty:
        return "", "text"

    file_name = matches.iloc[0]["file_name"]

    language = get_language(file_name)

    code = github.get_file_content(
        owner,
        repo,
        file_path
    )

    return code

###### 5. Gradio UI

In [188]:
with gr.Blocks(title="Code Explorer") as code_demo:

    gr.Markdown("# Code Explorer")
    gr.Markdown(
        "Select a file type, then select a file from the Neo4j codebase graph."
    )

    file_type_dropdown = gr.Dropdown(
        label="Select File Type",
        choices=get_file_types(),
        value=None,
        interactive=True
    )

    file_dropdown = gr.Dropdown(
        label="Select File",
        choices=[],
        value=None,
        interactive=True
    )

    code_output = gr.Code(
        label="Source Code",
        language="javascript",
        interactive=False,
        lines=40,
        max_lines=60
    )

    file_type_dropdown.change(
        fn=update_file_dropdown,
        inputs=file_type_dropdown,
        outputs=file_dropdown
    )

    file_dropdown.change(
        fn=get_selected_file,
        inputs=file_dropdown,
        outputs=code_output
    )

code_demo.launch()

* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.


`def get_file_content(self, owner, repo, file_path)`

In [184]:
github.get_file_content('georgejaymcmc', 'zotero', 'chrome/content/zotero-platform/unix/integration.css' )

'window.citation-dialog {\n\t-moz-appearance: none;\n\tbackground: -moz-linear-gradient(-90deg, rgb(243,123,119) 0, rgb(180,47,38) 50%, rgb(156,36,27) 50%) !important;\n}\n\nwindow.citation-dialog.note-dialog {\n\tbackground: -moz-linear-gradient(-90deg, rgb(249, 231, 179) 0, rgb(228, 193, 94) 50%, rgb(221, 184, 81) 50%) !important;\n}\n\n.citation-dialog.entry {\n\tpadding: 10px;\n}'

In [185]:
get_selected_file('chrome/content/zotero-platform/unix/integration.css')

('window.citation-dialog {\n\t-moz-appearance: none;\n\tbackground: -moz-linear-gradient(-90deg, rgb(243,123,119) 0, rgb(180,47,38) 50%, rgb(156,36,27) 50%) !important;\n}\n\nwindow.citation-dialog.note-dialog {\n\tbackground: -moz-linear-gradient(-90deg, rgb(249, 231, 179) 0, rgb(228, 193, 94) 50%, rgb(221, 184, 81) 50%) !important;\n}\n\n.citation-dialog.entry {\n\tpadding: 10px;\n}',
 'css')

In [104]:
import inspect

print(inspect.getsource(github._repo_data))
print(type(github._repo_data(owner, repo)))

    def _repo_data(self, owner, repo):
        """Get repository metadata, using the cache when available."""

        cache_key = f"{owner}/{repo}"

        if cache_key not in self._repo_cache:
            url = f"https://api.github.com/repos/{owner}/{repo}"

            response = requests.get(
                url,
                headers=self.headers,
                timeout=10
            )

            if response.status_code != 200:
                raise RuntimeError(
                    f"GitHub API error {response.status_code}: "
                    f"{response.text}"
                )

            self._repo_cache[cache_key] = response.json()

        return self._repo_cache[cache_key]

<class 'dict'>
